<a href="https://colab.research.google.com/github/ItsArnavRathi/Smart-student-feedback-insight-system/blob/main/Smart_Student_Feedback_Insight_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kagglehub

In [ ]:
import kagglehub

# Download dataset
path = kagglehub.dataset_download("omarsobhy14/university-students-complaints-and-reports")

print("Path to dataset files:", path)

100%|██████████| 38.8k/38.8k [00:00<00:00, 10.6MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/omarsobhy14/university-students-complaints-and-reports/versions/2


In [ ]:
import os

print(os.listdir(path))

['Datasetprojpowerbi.csv']


In [ ]:
import pandas as pd

file_path = path + "/Datasetprojpowerbi.csv"   # change this after checking
df = pd.read_csv(file_path)

df.head()

,Genre,Reports,Age,Gpa,Year,Count,Gender,Nationality
0,Academic Support and Resources,The limited access to research databases and m...,27,2.18,2,1,M,Egypt
1,Academic Support and Resources,I'm having trouble finding the course material...,23,3.11,2,1,F,Egypt
2,Academic Support and Resources,It's frustrating to have limited access to res...,20,3.68,2,1,F,Egypt
3,Academic Support and Resources,I'm really struggling in one of my classes but...,20,1.30,2,1,F,Egypt
4,Academic Support and Resources,I am really struggling with understanding the...,26,2.50,2,1,F,Egypt


In [ ]:
df = pd.read_csv(file_path)
print(df.columns)

Index(['Genre', 'Reports', 'Age', 'Gpa', 'Year', 'Count', 'Gender',
       'Nationality'],
      dtype='object')


In [ ]:
for col in df.columns:
    print(col)
    print(df[col].head(3))
    print("--------")

Genre
0    Academic Support and Resources
1    Academic Support and Resources
2    Academic Support and Resources
Name: Genre, dtype: object
--------
Reports
0    The limited access to research databases and m...
1    I'm having trouble finding the course material...
2    It's frustrating to have limited access to res...
Name: Reports, dtype: object
--------
Age
0    27
1    23
2    20
Name: Age, dtype: int64
--------
Gpa
0    2.18
1    3.11
2    3.68
Name: Gpa, dtype: float64
--------
Year
0    2
1    2
2    2
Name: Year, dtype: int64
--------
Count
0    1
1    1
2    1
Name: Count, dtype: int64
--------
Gender
0    M
1    F
2    F
Name: Gender, dtype: object
--------
Nationality
0    Egypt
1    Egypt
2    Egypt
Name: Nationality, dtype: object
--------


In [ ]:
df = df[['Reports']]

In [ ]:
df = df.dropna()

In [ ]:
df = df.head(100)

In [ ]:
df.shape
df.columns

Index(['Reports'], dtype='object')

In [ ]:
df.head(5)

,Reports
0,The limited access to research databases and m...
1,I'm having trouble finding the course material...
2,It's frustrating to have limited access to res...
3,I'm really struggling in one of my classes but...
4,I am really struggling with understanding the...


In [ ]:
df.isnull().sum()

,0
Reports,0


In [ ]:
df['Complaint_length'] = df['Reports'].apply(len)
df['Complaint_length'].describe()

,Complaint_length
count,100.000000
mean,136.180000
std,24.988636
min,67.000000
25%,118.750000
50%,134.000000
75%,153.250000
max,198.000000


Data Cleaning and Pre Processing

In [ ]:
import re

# ---- Keep only relevant columns ----
# Adjust column name if different in your file
df = df[['Reports']].copy()

# ---- Drop nulls ----
df.dropna(subset=['Reports'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Total records after cleaning: {len(df)}")

# ---- Text Cleaning Function ----
def clean_text(text):
    text = str(text).lower()                        # Lowercase
    text = re.sub(r'http\S+|www\S+', '', text)      # Remove URLs
    text = re.sub(r'[^a-z\s]', '', text)            # Remove numbers & special chars
    text = re.sub(r'\s+', ' ', text).strip()        # Remove extra spaces
    return text

# ---- Apply Cleaning ----
df['Cleaned_Report'] = df['Reports'].apply(clean_text)

# ---- Limit size for Colab (to avoid memory issues) ----
df = df.sample(n=min(500, len(df)), random_state=42).reset_index(drop=True)

print(f"\nWorking with {len(df)} samples")
print("\n--- Sample cleaned text ---")
print(df[['Reports', 'Cleaned_Report']].head(3))

Total records after cleaning: 100

Working with 100 samples

--- Sample cleaned text ---
                                             Reports  \
0  Limited course availability is making it impos...   
1  Limited access to technology and software is a...   
2  It's really frustrating to not have more acces...   

                                      Cleaned_Report  
0  limited course availability is making it impos...  
1  limited access to technology and software is a...  
2  its really frustrating to not have more access...  


In [ ]:
# =============================================
# STEP 3: SENTIMENT ANALYSIS (HuggingFace)
# =============================================

from transformers import pipeline

# ---- Load pre-trained sentiment model ----
# DistilBERT: lightweight BERT, fast on Colab, strong accuracy
print("Loading sentiment model... (this may take 1-2 minutes)")

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512
)

print("Model loaded successfully!")

# ---- Run sentiment analysis in batches ----
# Batching avoids memory crashes on Colab
def get_sentiment_batch(texts, batch_size=32):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size].tolist()
        batch_results = sentiment_pipeline(batch)
        results.extend(batch_results)
        print(f"  Processed {min(i+batch_size, len(texts))}/{len(texts)} rows...", end='\r')
    print("\nDone!")
    return results

# ---- Apply to cleaned text ----
print("\nRunning sentiment analysis...")
raw_results = get_sentiment_batch(df['Cleaned_Report'])

# ---- Extract label and score ----
df['Sentiment']       = [r['label'] for r in raw_results]   # POSITIVE / NEGATIVE
df['Sentiment_Score'] = [round(r['score'], 4) for r in raw_results]  # Confidence

# ---- Map to friendlier labels ----
df['Sentiment'] = df['Sentiment'].map({'POSITIVE': 'Positive', 'NEGATIVE': 'Negative'})

# ---- Preview results ----
print("\n--- Sample Sentiment Results ---")
print(df[['Cleaned_Report', 'Sentiment', 'Sentiment_Score']].head(6).to_string())

print("\n--- Sentiment Distribution ---")
print(df['Sentiment'].value_counts())